# Homework 3 - Adversarial Attacks and Contrastive Learning

### Submission Guidelines

- This notebook contains all the questions. You should follow the instructions below.
- Solutions for the practical parts should be written in this notebook/py (see submission files) and for the dry "theoretical" parts in a pdf report.

You should submit the following files:
- Code files - You can choose 1 of the following options:
  - IPYNB notebook including the solutions to all 4 parts. The `.ipynb` file should not include any answers to dry questions because they should all be in the PDF report. Markdown cells in the notebook will not be checked.
  - PY files: 2 `.py` files, one for questions 1-3 and one for question 4. Submission format for `.py` files -  `HW3_Partn_ID1_ID2.py` where `n=1` is for parts 1-3 and `n=2` is for part 4.

- PDF file - report:
  - Answers for all the dry parts - analysis, visualization, dry questions, embeddings. You must create a PDF report, even if you chose to implement your code using a Jupyter Notebook. The PDF should not be a PDF version of the notebook and doesn't need to include any code.

Good Luck!

Libraries that can be used in this assignment:

- PyTorch (torch, torchvision, torch.optim, torch.nn, ...) -- Any other deep learning libraries such as keras are prohibited
- NumPy
- Any libraries used for visualization
- PIL (for image processing)

For any libraries other than those listed above, you may ask in the forum.

## Part 1 - Training a CNN on SVHN (5pt)

Train a classifier on the dataset SVHN (Street View House Numbers), available through ```torchvision.datasets```. The model accuracy should reach at least 90% on the test set. You're free to add as many data augmentations as necessary. Take into consideration that any serious augmentations (like normalization) should be reversed when displaying images in future sections.

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
import torch.nn.functional as F


# Configuration
BATCH_SIZE = 64
LEARNING_RATE = 0.01
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SVHN Statistics for Normalization
MEAN = [0.4377, 0.4438, 0.4728]
STD = [0.1980, 0.2010, 0.1970]

print(f"Running on device: {DEVICE}")

### Data transformation

In [ ]:
# Data Augmentation: RandomCrop and Rotation (No Horizontal Flip for digits)
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Load SVHN Dataset
train_dataset = torchvision.datasets.SVHN(
    root='./data', split='train', download=True, transform=train_transform
)
test_dataset = torchvision.datasets.SVHN(
    root='./data', split='test', download=True, transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### Model Definition

In [ ]:
def get_svhn_resnet():
    # Load ResNet18 structure without pre-trained weights
    model = torchvision.models.resnet18(weights=None)

    # Replace the first 7x7 conv with 3x3, stride 1, padding 1
    # This prevents losing spatial info too early on small images
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # Remove the first maxpool
    model.maxpool = nn.Identity()

    # Adjust the final fully connected layer for 10 classes
    model.fc = nn.Linear(model.fc.in_features, 10)

    return model

model = get_svhn_resnet().to(DEVICE)

# Setup Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

### Visualization Utilities

In [ ]:
class UnNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        """
        Args:
            tensor (Tensor): Tensor image of size (C, H, W) to be un-normalized.
        Returns:
            Tensor: Un-normalized image.
        """
        tensor = tensor.clone() # Avoid modifying the original tensor
        for t, m, s in zip(tensor, self.mean, self.std):
            t.mul_(s).add_(m)
        return tensor

# Initialize the un-normalizer
unorm = UnNormalize(mean=MEAN, std=STD)

def imshow(tensor):
    """Helper to display a tensor as an image"""
    image = unorm(tensor)
    # Move to cpu and numpy, transpose from (C, H, W) to (H, W, C) for matplotlib
    np_img = image.cpu().numpy().transpose((1, 2, 0))
    plt.imshow(np_img)
    plt.axis('off')

# Quick test to see a batch
dataiter = iter(train_loader)
images, labels = next(dataiter)

plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    imshow(images[i])
    plt.title(f"Label: {labels[i].item()}")
plt.show()

### Train & Eval

In [ ]:
def train_one_epoch(epoch_index):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    acc = 100. * correct / total
    print(f"Epoch [{epoch_index+1}/{EPOCHS}] Loss: {avg_loss:.4f} | Train Acc: {acc:.2f}%")

def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc

### Execution

In [ ]:
print(f"Starting training on {DEVICE}...")

train_acc_history = []
test_acc_history = []

for epoch in range(EPOCHS):
    train_one_epoch(epoch)
    acc = evaluate()
    test_acc_history.append(acc)
    scheduler.step()

print("Training Complete.")

### Analysis

Analyze the performance of the model on the test set (e.g. through a confusion matrix). Display at least 4 images that the model predicts incorrectly and their predicted classes. Discuss possible weaknesses of the model and their causes.

### Confusion matrix

In [ ]:
def get_all_preds(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    misclassified = [] # Store tuple: (image_tensor, true_label, pred_label)

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Identify misclassified images in this batch
            mask = preds != labels
            if mask.any():
                wrong_imgs = inputs[mask]
                wrong_preds = preds[mask]
                true_lbls = labels[mask]
                for img, t, p in zip(wrong_imgs, true_lbls, wrong_preds):
                    # Store only a few to save memory, or all if dataset is small enough
                    if len(misclassified) < 20:
                        misclassified.append((img.cpu(), t.item(), p.item()))

    return np.array(all_preds), np.array(all_labels), misclassified

# Get predictions
test_preds, test_labels, misclassified_examples = get_all_preds(model, test_loader)

# Generate Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: SVHN Test Set')
plt.show()

### Show misclassified images

In [ ]:
# Select the first 5 misclassified examples
num_show = 5
fig, axes = plt.subplots(1, num_show, figsize=(15, 4))

for i in range(num_show):
    img_tensor, true_label, pred_label = misclassified_examples[i]

    # Un-normalize and prepare for plotting
    img = unorm(img_tensor)
    img_np = img.numpy().transpose((1, 2, 0))
    # Clip to valid range [0, 1] to avoid matplotlib warnings if un-norm drifts slightly
    img_np = np.clip(img_np, 0, 1)

    ax = axes[i]
    ax.imshow(img_np)
    ax.set_title(f"True: {true_label} | Pred: {pred_label}", color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

### Possible weasknesses:

1. The model can missclassify house numbers which there's a cropped number in the edge
    * Why?: the model sees some feautured of the cropped nuber on the side and gets confused. For example, if the house number is 12 and the label is 2, the 1 might be still in the picture, confusing the model.

2. The model is wrong more frequently on blury pictures.
    * Why?: on blury pictures its hard to identify which number is that. like 8 and 9 are very simmilar on blury pictures

## Part 2: Adversarial Attacks on our Model (15pt)

### Dry Question
In Slide 14 of the tutorial on adversarial attacks, we showed how attacks can be transfered from one model to another, i.e. if we craft an attack for ResNet50 using FGSM, it will likely also confuse other models. Why is this property important, specifically when attacking an unknown model?

### Why is this important?
  * Now the attacker doesnt need access to the network parameters like we needed in FGSM. We can rely only on the black box attack which is much easier.

  * Another big issue is, because the attack is transferable, the attacker might as well train a neural network (for example ResNet) on his pc and craft an attack and use it on every other network, this is a big problem!


Implement the FGSM algorithm as shown in the tutorial (it may be necessary to make require_grad be True for our input data in order to calculate its gradient). Define a function ```eval_adversarial(model, test_loader, epsilon)``` that applies FGSM on the test data by creating new images which are perturbed versions of the originals, and calculates the model's accuracy only on the perturbed images. Test accuracy after the attack should be less than 25% for $\varepsilon=0.1$.

### The attack

In [ ]:
from attacks import fgsm_attack
# epsilon remains in normalized-input units; the default now clips to valid pixel bounds.
# For historical unconstrained perturbations, explicitly pass clamp=False.


### Evaluation

In [ ]:
def eval_adversarial(model, test_loader, epsilon):
    """
    Applies FGSM on the test data and calculates accuracy on the perturbed images.
    """
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0

    # Iterate over the test set
    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)

        # Enable gradient calculation for the input data
        data.requires_grad = True

        # Forward pass the data through the model
        output = model(data)

        # Calculate the loss
        loss = F.cross_entropy(output, target)

        # Zero all existing gradients
        model.zero_grad()

        # Calculate gradients of model in backward pass
        loss.backward()

        data_grad = data.grad.data

        # Call FGSM Attack to create the perturbed image
        perturbed_data = fgsm_attack(data, epsilon, data_grad).detach()

        # Re classify the perturbed image
        output_adv = model(perturbed_data)

        # Check for success
        _, final_pred = torch.max(output_adv, 1)
        total += target.size(0)
        correct += (final_pred == target).sum().item()

    # Calculate final accuracy
    final_acc = correct / total
    print(f"Epsilon: {epsilon}\tTest Accuracy (Adversarial): {final_acc * 100:.2f}%")

    return final_acc

### Execution

In [ ]:
# Define the epsilon value
epsilon = 0.1

print(f"Starting FGSM attack with epsilon = {epsilon}...")

# Run the evaluation
adv_accuracy = eval_adversarial(model, test_loader, epsilon)

print(f"Accuracy: {adv_accuracy*100:.2f}%)")

### Visualization

Visualize some images that the model got right before the perturbation and wrong after the attack. Create a confusion matrix of the output on the entire test set (looking only at the perturbed images).

In [ ]:
def analyze_adversarial_performance(model, loader, epsilon):
    model.eval()
    all_adv_preds = []
    all_true_labels = []
    successful_attacks = []  # Stores (original_image, adv_image, true_label, orig_pred, adv_pred)

    # We only need a few examples for visualization
    max_examples = 5

    for data, target in loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        data.requires_grad = True

        # Forward pass on ORIGINAL data
        output_orig = model(data)
        _, init_pred = output_orig.max(1)

        # Create a mask for correctly classified images
        mask_correct = init_pred == target

        if not mask_correct.any():
            continue

        # Calculate Loss and Gradients
        loss = F.cross_entropy(output_orig, target)
        model.zero_grad()
        loss.backward()
        data_grad = data.grad.data

        # Generate adversarial Images
        perturbed_data = fgsm_attack(data, epsilon, data_grad)

        # Forward pass on adversarial data
        output_adv = model(perturbed_data)
        _, adv_pred = output_adv.max(1)

        # Collect Data for Confusion Matrix
        all_adv_preds.extend(adv_pred.cpu().detach().numpy())
        all_true_labels.extend(target.cpu().detach().numpy())

        # Find Successful Attacks for Visualization
        mask_attack_success = (adv_pred != target) & mask_correct

        if len(successful_attacks) < max_examples and mask_attack_success.any():
            # Get indices where attack succeeded
            idxs = torch.nonzero(mask_attack_success).flatten()
            for idx in idxs:
                if len(successful_attacks) < max_examples:
                    successful_attacks.append((
                        data[idx].detach().cpu(),
                        perturbed_data[idx].detach().cpu(),
                        target[idx].item(),
                        init_pred[idx].item(),
                        adv_pred[idx].item()
                    ))

    return np.array(all_adv_preds), np.array(all_true_labels), successful_attacks

# Run the analysis
print(f"Analyzing adversarial attacks with epsilon={epsilon}...")
adv_preds, true_labels, success_examples = analyze_adversarial_performance(model, test_loader, epsilon)

### Visualization

In [ ]:
# Successful Attacks
print(f"Displaying {len(success_examples)} examples where the model was fooled:")

plt.figure(figsize=(15, 6))
for i, (orig_img, adv_img, true_lbl, orig_pred, adv_pred) in enumerate(success_examples):
    # Top Row: Original Images
    plt.subplot(2, 5, i + 1)

    plt.imshow(unorm(orig_img).permute(1, 2, 0).clip(0, 1))
    plt.title(f"Original: {true_lbl}\nPred: {orig_pred}", fontsize=10)
    plt.axis('off')

    # Bottom Row: Adversarial Images
    plt.subplot(2, 5, i + 6)
    plt.imshow(unorm(adv_img).permute(1, 2, 0).clip(0, 1))
    plt.title(f"Adversarial\nPred: {adv_pred}", color='red', fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

# Confusion Matrix (Adversarial Data)
print("Generating Confusion Matrix for the Adversarial Test Set...")

cm = confusion_matrix(true_labels, adv_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False)
plt.xlabel('Predicted Label (Adversarial)')
plt.ylabel('True Label')
plt.title(f'Confusion Matrix: FGSM Attack (Epsilon={epsilon})')
plt.show()

Test the function with different values of epsilon (at least 5) and plot the accuracy as a function of epsilon. For each epsilon, display the perturbed images with the model's classification. At what epsilon does it become harder for the human eye to correctly classify?

### Epsilon testing

In [ ]:
# Define the list of epsilon values to test
epsilons = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
accuracies = []
examples = []

# Select a fixed batch of data to visualize consistency across epsilons
# We take the first 5 images from the test loader
fixed_data, fixed_target = next(iter(test_loader))
fixed_data, fixed_target = fixed_data.to(DEVICE), fixed_target.to(DEVICE)
fixed_data = fixed_data[:5]
fixed_target = fixed_target[:5]

print("Running FGSM Sweep...")

for eps in epsilons:
    # Calculate Accuracy on the full test set
    acc = eval_adversarial(model, test_loader, eps)
    accuracies.append(acc)

    # Enable grad for the fixed data
    data_temp = fixed_data.clone().detach()
    data_temp.requires_grad = True

    output = model(data_temp)
    loss = criterion(output, fixed_target)
    model.zero_grad()
    loss.backward()
    data_grad = data_temp.grad.data

    # Generate perturbed images
    pert_imgs = fgsm_attack(data_temp, eps, data_grad).detach()

    # Get the model's prediction on these perturbed images
    with torch.no_grad():
        output_adv = model(pert_imgs)
        _, preds_adv = output_adv.max(1)

    # Store for plotting
    examples.append((eps, pert_imgs.cpu(), preds_adv.cpu()))

print("Sweep Complete.")

### Plotting accuracy vs. Epsilon

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epsilons, accuracies, "*-")
plt.yticks(np.arange(0, 1.1, step=0.1))
plt.xticks(epsilons)
plt.title("Accuracy vs Epsilon (FGSM Attack)")
plt.xlabel("Epsilon")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

### Visualization

In [ ]:
# Number of epsilons (rows) and examples (columns)
n_rows = len(epsilons)
n_cols = len(fixed_data)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 15))

for i, (eps, pert_imgs, preds) in enumerate(examples):
    for j in range(n_cols):
        ax = axes[i, j]

        # Process image for display
        img = unorm(pert_imgs[j])
        img = img.permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)

        ax.imshow(img)

        # Color the title red if wrong, black if correct
        true_label = fixed_target[j].item()
        pred_label = preds[j].item()
        color = 'black' if true_label == pred_label else 'red'

        # Only show row labels on the first column
        if j == 0:
            ax.set_ylabel(f"Eps: {eps}", fontsize=14, fontweight='bold')

        ax.set_title(f"Pred: {pred_label} (True: {true_label})", color=color, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

## Part 3: Training our model using adversarial training (20pt)

For each point in the training data, increase the model's robustness by training not only on the point itself, but on the perturbed point after the FGSM algorithm using $\varepsilon=0.1$. Afterwards, compute the accuracy once again on the newly trained model using ```eval_adversarial(model, test_loader, epsilon)``` defined above. The accuracy **looking only at the perterubed data** should be at least 70%.

### Train function

In [ ]:
def train_adversarial_epoch(model, loader, optimizer, epsilon, alpha=0.5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        inputs.requires_grad = True

        # Forward pass to determine the direction of the gradient
        outputs_orig = model(inputs)
        loss_orig = criterion(outputs_orig, labels)

        # clear previous gradients
        model.zero_grad()
        loss_orig.backward()

        # Get the gradient of the loss w.r.t. the input data
        data_grad = inputs.grad.data

        # Create the adversarial examples using FGSM
        perturbed_inputs = fgsm_attack(inputs, epsilon, data_grad).detach()

        # Clear gradients again for the actual weight update
        optimizer.zero_grad()
        inputs.requires_grad = False # Turn off for the weight update step

        # Forward pass: Clean Data
        outputs_clean = model(inputs)
        loss_clean = criterion(outputs_clean, labels)

        # Forward pass: Adversarial Data
        outputs_adv = model(perturbed_inputs)
        loss_adv = criterion(outputs_adv, labels)

        # Combined Loss
        total_loss = alpha * loss_clean + (1 - alpha) * loss_adv

        total_loss.backward()
        optimizer.step()

        # Track stats
        running_loss += total_loss.item()
        _, predicted = outputs_clean.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = running_loss / len(loader)
    acc = 100. * correct / total
    print(f"Adv Train Epoch Loss: {avg_loss:.4f} | Clean Acc: {acc:.2f}%")

### Exectution

In [ ]:
# Configuration for Adversarial Training
ADV_EPOCHS = 15
epsilon_train = 0.1

print(f"Starting Adversarial Training (epsilon={epsilon_train})...")

# We can lower the learning rate slightly for fine-tuning
optimizer_adv = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(ADV_EPOCHS):
    print(f"Epoch {epoch+1}/{ADV_EPOCHS}")
    train_adversarial_epoch(model, train_loader, optimizer_adv, epsilon_train)

print("Adversarial Training Complete.")

### Evaluation

In [ ]:
print("Evaluating Robustness on Adversarial Test Set...")

# Evaluate using the same epsilon used for training
# The requirement is that accuracy on perturbed data > 70%
final_adv_acc = eval_adversarial(model, test_loader, epsilon=0.1)

print(f"Accuracy: {final_adv_acc*100:.2f}%")

### Visualization

Display the confusion matrix along with some examples of images that the model classified incorrectly. Discuss the performance of the model now compared to before.

In [ ]:
print("Analyzing Robust Model Performance...")

# Re run the analysis with the same epsilon
adv_preds_robust, true_labels_robust, remaining_failures = analyze_adversarial_performance(model, test_loader, epsilon=0.1)

# Confusion Matrix
cm_robust = confusion_matrix(true_labels_robust, adv_preds_robust)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_robust, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.xlabel('Predicted Label (Robust Model)')
plt.ylabel('True Label')
plt.title('Confusion Matrix: Robust Model on Adversarial Data')
plt.show()

# Visualize Remaining Failures
if len(remaining_failures) > 0:
    print(f"Displaying {len(remaining_failures)} examples that still fool the model:")
    plt.figure(figsize=(15, 4))
    for i, (orig_img, adv_img, true_lbl, orig_pred, adv_pred) in enumerate(remaining_failures):
        if i >= 5: break # Show max 5

        plt.subplot(1, 5, i + 1)
        # Show the adversarial image that caused the error
        plt.imshow(unorm(adv_img).permute(1, 2, 0).clip(0, 1))
        plt.title(f"True: {true_lbl}\nPred: {adv_pred}", color='red', fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Amazing! The model correctly classified all adversarial examples in the visualization batch.")

### Discussion: Performance Comparison

1. Precentages change:
  * Before: The model was very bad. With $\epsilon=0.1$, the accuracy likely dropped from around 90% (clean) to 25%  (adversarial). This indicates that the model's decision boundaries were sharp and close to the data points.

  * After: The accuracy on is now on 70\%. While this is likely lower than the original clean accuracy, it's a massive improvement in robustness. The model is no longer surprised by attacks like FGSM.


2. The Confusion Matrix:

  * Before: The matrix showed scattered predictions, indicating the gradient noise was successfully pushing images into specific, unrelated parts of the feature space.

  * After: The matrix look much closer to a standard diagonal. The errors that remain are probably just normal errors or very blury images, like the ones we see above.


## Part 4: Contrastive Learning

### **Dry Questions**

1.


We prefer a large batch size. The reason is that in SimCLR, all other images in the batch act as "negative" examples for the specific image we are training on. When the batch is large, the model has to distinguish the correct image from many other negative examples. This makes the task harder, which forces the model to learn much better features.

2.

In training: We apply Random Augmentations (like cropping, color jitter) to create two different versions of the same image.We use the output of the Projection Head ($z$) to calculate the contrastive loss.

In testing :We do not use random augmentations (only standard Resize and Normalize).We usually use the output of the Encoder ($h$, before the projection head) as the final embedding, because it contains more informative features.

3.

- Randomly cropping a fixed-size window: No

 SimCLR specifically relies on Random Resized Crop (cropping a window of random size and resizing it). Cropping a fixed size window isn't enough because it doesn't teach the model Scale Invariance (recognizing the object from different distances/scales).

- Enlarging to 128x128: No.

 Tiny ImageNet is 64x64. Upscaling adds computational cost (slower training) without adding any new information or features.

- Random rotation: No.

 Rotation does not improve performance for natural images (like ImageNet), because usually, the orientation matters (e.g., the sky is up, trees grow up). It makes the task too hard or irrelevant.

- Adding Gaussian Noise: No.

 SimCLR uses Gaussian Blur, not noise. We want the model to ignore high-frequency details (like texture/noise) and focus on the shape.

- Randomly changing the image's dimensions: No.

 We need all images in a Batch to have the same dimensions (e.g., 64x64) to stack them into a Tensor.

- Randomly converting to grayscale: Yes.

 Without it, the model can "cheat" by just matching the color distribution (histograms) of the two images instead of learning the actual content.

### **Data Extraction and Path Configuration**

In [ ]:
!unzip -q -o /content/drive/MyDrive/archive.zip

In [ ]:
import os
import torch
from torch.utils.data import DataLoader


# 2. Setup Paths
base_dir = os.path.abspath("tiny-imagenet-200")
TRAIN_PATH = os.path.join(base_dir, 'train')
TEST_PATH = os.path.join(base_dir, 'test')


print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"TEST_PATH:  {TEST_PATH}")


### **Random Seed**

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    # 1. Set seed for Python's built-in random module
    random.seed(seed)

    # 2. Set seed for NumPy
    np.random.seed(seed)

    # 3. Set seed for PyTorch (CPU)
    torch.manual_seed(seed)

    # 4. Set seed for PyTorch (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # If using multi-GPU

        # 5. Ensure deterministic behavior for cuDNN (optional but recommended)
        # These settings might slow down training slightly but ensure exact reproducibility
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"Random seed set to: {seed}")

# Usage
set_seed(42)

We set a global random seed to ensure reproducibility. Deep learning models rely on stochastic processes such as random weight initialization, data shuffling, and augmentations. Without a fixed seed, every training run would yield different results solely due to chance. By fixing the seed, we isolate the randomness, ensuring that any difference in performance is caused by our specific changes to the model or hyperparameters. This allows us to reliably analyze and compare the results

### **Data Transformation**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import numpy as np

# --- Configuration ---
IMAGE_SIZE = 64  # Tiny ImageNet usually works well with 64x64
BATCH_SIZE = 6144 # Larger batch sizes are generally better for Contrastive Learning

# --- Data Augmentations ---

# SimCLR requires strong color jittering
s = 1.0
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)

# Training transforms: Aggressive augmentations to create two different views
data_transforms = transforms.Compose([
    # Randomly crop and resize the image. This forces the model to learn local and global features.
    transforms.RandomResizedCrop(size=IMAGE_SIZE, scale=(0.08, 1.0)),

    # Randomly flip the image horizontally
    transforms.RandomHorizontalFlip(),

    # Randomly apply color jittering (brightness, contrast, saturation, hue)
    # Applied with probability 0.8
    transforms.RandomApply([color_jitter], p=0.8),

    # Randomly convert to grayscale.
    # This prevents the model from relying solely on color histograms.
    transforms.RandomGrayscale(p=0.2),

    # Convert to PyTorch Tensor
    transforms.ToTensor(),

    # Normalize (optional but recommended for stability)
    transforms.Normalize(mean=mean, std=std)
])

# Test transforms: No augmentations, just resizing and converting to tensor
test_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


### **Custom SimCLR Dataset and Train Data Loading**




In [ ]:
import os
from PIL import Image
from torch.utils.data import DataLoader
import torch
from torchvision import transforms

class TrainDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.imgs = []

        if not os.path.exists(root):
            raise ValueError(f"The path {root} does not exist! Please check TRAIN_PATH.")

        for label in os.listdir(root):
            # Ensure we are looking at a directory
            label_path = os.path.join(root, label)
            if os.path.isdir(label_path):
                images_dir = os.path.join(label_path, 'images')
                if os.path.isdir(images_dir):
                    for img in os.listdir(images_dir):
                        self.imgs.append(os.path.join(images_dir, img))
                else:
                    # Fallback in case the structure is different (e.g., standard ImageNet structure)
                    for img in os.listdir(label_path):
                         if img.endswith(('.jpg', '.png', '.jpeg')):
                            self.imgs.append(os.path.join(label_path, img))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        # Open image and convert to RGB (crucial for handling grayscale inputs consistently)
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            # The Magic happens here: Apply the transform TWICE to get two different views
            img1 = self.transform(img)
            img2 = self.transform(img)
        else:
            img1, img2 = img, img

        return img1, img2

# Create the DataLoader
# We use the BATCH_SIZE defined in the configuration block (128) and the data_transforms
print(f"Loading data from: {TRAIN_PATH}")
train_data = TrainDataset(TRAIN_PATH, transform=data_transforms)

# drop_last=True is important in Contrastive Learning to avoid getting a small batch at the end,
# which could make the loss calculation unstable due to too few negative samples.
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=16)

print(f"Data loaded successfully. Number of batches: {len(train_loader)}")

### **Data Visualization**

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np
from torchvision import transforms

# --- Helper Configuration ---
# Standard ImageNet statistics (must match what you used in transforms)

NUM_IMAGES_TO_SHOW = 4

# --- Define unnormalize locally to fix NameError ---
def unnormalize(tensor, mean=mean, std=std):
    """
    Reverses the normalization to display the image correctly.
    """
    # Inverse math: (x * std) + mean
    inverse_mean = [-m/s for m, s in zip(mean, std)]
    inverse_std = [1/s for s in std]

    unorm_transform = transforms.Normalize(mean=inverse_mean, std=inverse_std)
    return unorm_transform(tensor)

# --- Sanity Check Function ---
def show_batch_sanity_check(loader):
    # 1. Extract: Get a single batch
    try:
        img1_batch, img2_batch = next(iter(loader))
    except NameError:
        print("Error: train_loader is not defined. Please run the Data Loading cell first.")
        return

    # 2. Print Basic Stats
    print(f"Batch Shapes: {img1_batch.shape}")
    print(f"Tensor Type: {img1_batch.dtype}")
    # Note: If Min is 0.0 and Max is 1.0, Normalization was NOT applied in transforms!
    print(f"Min Value: {img1_batch.min():.2f}")
    print(f"Max Value: {img1_batch.max():.2f}")

    # 3. Visualize
    fig, axes = plt.subplots(NUM_IMAGES_TO_SHOW, 2, figsize=(10, NUM_IMAGES_TO_SHOW * 3))
    plt.subplots_adjust(wspace=0.1, hspace=0.2)

    for i in range(NUM_IMAGES_TO_SHOW):
        # A. Unnormalize (restore original colors)
        # If your data is 0-1 (not normalized), unnormalize might make it bright, but that's okay for checking
        v1 = unnormalize(img1_batch[i])
        v2 = unnormalize(img2_batch[i])

        # B. Clamp values to [0, 1] to avoid warnings
        v1 = torch.clamp(v1, 0, 1)
        v2 = torch.clamp(v2, 0, 1)

        # C. Convert to Numpy for plotting
        v1_np = v1.permute(1, 2, 0).numpy()
        v2_np = v2.permute(1, 2, 0).numpy()

        # D. Plot
        axes[i, 0].imshow(v1_np)
        axes[i, 0].set_title(f"Sample {i} - View 1")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(v2_np)
        axes[i, 1].set_title(f"Sample {i} - View 2")
        axes[i, 1].axis('off')

    plt.suptitle("SimCLR Positive Pairs (Augmented Views)", fontsize=16)
    plt.show()

# Run the check
print("Running ETL Sanity Check...")
show_batch_sanity_check(train_loader)

### **SimCLR Model Architecture: Encoder & Projection Head**

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, base_model=None, out_dim=128):
        """
        Args:
            base_model: The base encoder architecture (default: ResNet18).
            out_dim: The dimension of the projection head output (z).
        """
        super(SimCLR, self).__init__()

        # 1. Initialize the Encoder (Base Model)
        if base_model is None:
            # Load ResNet18 without pretrained weights (we want to learn from scratch)
            self.encoder = models.resnet18(pretrained=False)
        else:
            self.encoder = base_model

        # Get the input dimension of the last layer (fc) of ResNet
        self.feature_dim = self.encoder.fc.in_features

        # Replace the classification layer with Identity to extract features (h)
        self.encoder.fc = nn.Identity()

        # 2. Initialize the Projection Head
        # The projection head maps the representation h to the space z where loss is calculated.
        # Architecture: Linear -> ReLU -> Linear
        self.projection_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        """
        Returns:
            h: The representation (used for downstream tasks).
            z: The projection (used for contrastive loss calculation).
        """
        # Get the feature vector h from the encoder
        h = self.encoder(x)

        # Get the projection z from the head
        z = self.projection_head(h)

        return h, z

### **NT-Xent Loss Implementation**

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.07):
    """
    Calculates the NT-Xent loss (Normalized Temperature-scaled Cross Entropy Loss).

    Args:
        z1: Projections of the first view of the batch. Shape: [Batch_Size, out_dim]
        z2: Projections of the second view of the batch. Shape: [Batch_Size, out_dim]
        temperature: Temperature parameter to scale the cosine similarity.
    """
    batch_size = z1.shape[0]

    # 1. Concatenate the two views to process them together
    out = torch.cat([z1, z2], dim=0) # Shape: [2*N, out_dim]

    # 2. Normalize the vectors (L2 norm) so that dot product equals cosine similarity
    out = nn.functional.normalize(out, dim=1)

    # 3. Calculate Similarity Matrix
    # Matrix multiplication of 'out' and its transpose gives cosine similarity between all pairs
    sim_matrix = torch.mm(out, out.t()) / temperature # Shape: [2*N, 2*N]

    # 4. Mask out the diagonal (self-similarity)
    # We don't want the model to learn that an image is similar to itself (trivial)
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(out.device)

    # Fill the diagonal with a very small number so Softmax will zero it out
    sim_matrix.masked_fill_(mask, -9e15)

    # 5. Create Labels
    # For a sample i (in the first half), the positive pair is at index (i + batch_size)
    # For a sample i+batch_size (in the second half), the positive pair is at index i
    labels = torch.cat([
        torch.arange(batch_size) + batch_size,
        torch.arange(batch_size)
    ]).to(out.device)

    # 6. Calculate Cross Entropy Loss
    loss = nn.functional.cross_entropy(sim_matrix, labels)

    return loss

### **SimCLR Training**

In [ ]:
import torch
import torch.optim as optim
import matplotlib.pyplot as plt  # Import matplotlib for plotting

# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize Model
# Assuming SimCLR class is defined elsewhere
model = SimCLR().to(device)

# Initialize Optimizer
optimizer = optim.Adam(model.parameters(), lr=6e-3, weight_decay=1e-4)

# Training Parameters
EPOCHS = 250

# List to store loss values for plotting
loss_history = []

# --- Training Loop ---
model.train()

print("Starting training...")
for epoch in range(EPOCHS):
    total_loss = 0

    # Iterate over the training loader
    for img1, img2 in train_loader:
        img1, img2 = img1.to(device), img2.to(device)

        optimizer.zero_grad()

        _, z1 = model(img1)
        _, z2 = model(img2)

        # Assuming nt_xent_loss is defined elsewhere
        loss = nt_xent_loss(z1, z2, temperature=0.07)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Calculate average loss for the epoch
    avg_loss = total_loss / len(train_loader)

    # Store the loss value
    loss_history.append(avg_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

print("Training finished.")

# --- Plotting the Graph ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), loss_history, label='Training Loss', color='blue')
plt.title('SimCLR Contrastive Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()


### **Test Data Loading for SimCLR**

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from PIL import Image
from torchvision import transforms

# --- 1. Define TEST_PATH ---
# We assume 'base_dir' was defined in previous steps (e.g., 'tiny-imagenet-200')
# If not, please replace base_dir with the actual path string.
TEST_PATH = os.path.join(base_dir, 'test')
print(f"Defining TEST_PATH as: {TEST_PATH}")

# --- 2. Robust Dataset Class for Test Data ---
class RobustTestDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.imgs = []

        # Supported extensions (including uppercase)
        valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG', '.PNG')

        print(f"Scanning for Test images in: {root}")

        if not os.path.exists(root):
            print(f"❌ Error: The path {root} does not exist!")
            return

        # Check if standard 'images' subdirectory exists
        images_dir = os.path.join(root, 'images')

        if os.path.isdir(images_dir):
            print(f"Found 'images' subdirectory. Scanning inside...")
            search_dir = images_dir
        else:
            # If no 'images' folder, scan the root (flat directory structure)
            print(f"No 'images' subdirectory. Scanning root: {root}")
            search_dir = root

        # Scan files
        # This handles both flat lists of images and class folders
        for item in os.listdir(search_dir):
            item_path = os.path.join(search_dir, item)

            # If it's an image file
            if os.path.isfile(item_path) and item.endswith(valid_exts):
                self.imgs.append(item_path)
            # If it's a directory (class folder), scan inside
            elif os.path.isdir(item_path):
                for img in os.listdir(item_path):
                    if img.endswith(valid_exts):
                        self.imgs.append(os.path.join(item_path, img))

        # Sort to ensure consistent order/reproducibility
        self.imgs.sort()

        print(f"Found {len(self.imgs)} test images.")

        if len(self.imgs) == 0:
            raise ValueError(f"No images found in {search_dir}. Please check the path.")

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]

        # Open image and convert to RGB (prevents errors with grayscale images)
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a black placeholder image in case of error
            img = Image.new('RGB', (64, 64))

        if self.transform:
            img_transform = self.transform(img)
        else:
            img_transform = transforms.ToTensor()(img)

        # Returns:
        # 1. Original Tensor (for visualization)
        # 2. Transformed Tensor (for the model input)
        return transforms.ToTensor()(img), img_transform

# --- 3. Create the Loader ---
# Ensure test_transforms is defined
if 'test_transforms' not in globals():
    test_transforms = transforms.Compose([
        transforms.Resize(64),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

try:
    test_dataset = RobustTestDataset(TEST_PATH, transform=test_transforms)
    # Shuffle is usually False for testing, but useful here for random visualization samples
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)
    print("✅ Robust Test Loader created successfully!")
except ValueError as e:
    print(f"❌ Error: {e}")

In [ ]:
# Verify the batch size of the test_loader
if 'test_loader' in locals():
    print(f"Test Loader Batch Size: {test_loader.batch_size}")
elif 'BATCH_SIZE' in locals():
    print(f"test_loader is not defined yet, but the global BATCH_SIZE is configured to: {BATCH_SIZE}")
else:
    print("BATCH_SIZE variable is not defined.")


### **t-SNE Visualization of SimCLR Embeddings**

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import numpy as np

def plot_embeddings(model, test_loader, device, num_images=50, zoom=0.5):
    """
    Projects embeddings to 2D using t-SNE and plots them.
    Robustly handles batch sizes and perplexity constraints.
    """
    model.eval()

    # 1. Get a single batch
    try:
        # Use next(iter()) to safely get the first batch
        img_original, img_transform = next(iter(test_loader))
    except StopIteration:
        print("Loader is empty.")
        return

    # 2. Limit the number of images to avoid overplotting
    # Take the smaller of the actual batch size or the requested limit
    actual_limit = min(len(img_original), num_images)

    img_original = img_original[:actual_limit]
    img_transform = img_transform[:actual_limit]

    img_transform = img_transform.to(device)

    # 3. Get Embeddings
    with torch.no_grad():
        output = model(img_transform)
        # Handle cases where model returns a tuple (e.g., (embeddings, logits))
        if isinstance(output, tuple) or isinstance(output, list):
            embeddings = output[0]
        else:
            embeddings = output

    # 4. Run t-SNE
    # CRITICAL FIX: Calculate perplexity based on the ACTUAL number of samples
    n_samples = embeddings.shape[0]

    # Perplexity must be less than n_samples.
    # We aim for 30 (standard default), but cap it at n_samples - 1.
    perplexity_val = min(30, n_samples - 1)

    # Safety check: perplexity cannot be less than 1
    if perplexity_val < 1:
        perplexity_val = 1

    print(f"Running t-SNE on {n_samples} samples with perplexity={perplexity_val}...")

    tsne = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=perplexity_val)
    embeddings_2d = tsne.fit_transform(embeddings.cpu().numpy())

    # 5. Plotting
    fig, ax = plt.subplots(figsize=(12, 12)) # Larger figure size for better visibility

    # Plot faint dots first to see the structure even if images overlap
    ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.3)

    for i in range(embeddings_2d.shape[0]):
        # Prepare image for Matplotlib (C, H, W) -> (H, W, C)
        curr_img = img_original[i].permute(1, 2, 0).numpy()

        # Clip values to 0-1 range just in case of normalization issues
        curr_img = np.clip(curr_img, 0, 1)

        # Create the image annotation
        im = OffsetImage(curr_img, zoom=zoom)
        ab = AnnotationBbox(im, embeddings_2d[i], xycoords='data', frameon=False)
        ax.add_artist(ab)

    # 6. Dynamic Limits
    # Instead of hardcoded limits, we let matplotlib decide based on data
    ax.autoscale()

    # Optional: Remove axes for a cleaner "artistic" look
    plt.axis('off')
    plt.title(f"t-SNE Visualization of {n_samples} Images")
    plt.show()

# Usage Example:
plot_embeddings(model, test_loader, device, num_images=40, zoom=0.6)


This t-SNE plot projects your model's high-dimensional understanding of images down to a 2D map. The distance represents similarity: images close together are seen as similar by the model, while distant ones are distinct.

- **Model Verdict:** The model is learning effectively, primarily clustering by color and texture:

    - Blue/Water (Bottom-Left): Successfully grouped water, sky, and blue-tinted scenes.

    - Earth-Tones (Right): Grouped brownish textures and animals together. The distinct groupings prove the model is encoding visual features and not outputting random noise.

- **Code Improvements**
To generate this clear visualization, we made the following technical adjustments:

    - Dynamic Scaling (ax.autoscale): Replaced hardcoded limits (-5, 5) which were cutting off data, ensuring all clusters are visible regardless of the t-SNE output range.

    - Batch Slicing ([:num_images]): Limited the plot to ~40-50 images (subset of the 256 batch) to prevent "overplotting" and ensure individual images are distinguishable.

    - Robust Output Handling: Added a tuple check (isinstance) to safely extract embeddings, preventing crashes if the model returns extra data (like classification logits).

    - Enhanced Aesthetics: Increased figsize to (12, 12) and removed axes (plt.axis('off')) to focus attention entirely on the semantic clusters rather than arbitrary coordinates.


### **Evaluation**

#### 1. ####

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import os
from PIL import Image
import numpy as np
from torchvision import transforms

def plot_two_classes_separation(model, dataset_root, transform, device):
    """
    Plots t-SNE of 20 images from two specific classes to check separation.
    Uses direct file access since the Unsupervised Dataset doesn't return labels.
    """
    model.eval()

    # 1. Find two different classes (folders)
    # In Tiny ImageNet the structure is usually: train/class_id/images/img.jpg
    classes = [d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]
    classes.sort()

    if len(classes) < 2:
        print("Error: Not enough classes found in dataset.")
        return

    Fish = classes[0] # The first class found
    Crocodile = classes[4] # The second class found (you can change indices to pick others)

    print(f"Selected Class A: Fish")
    print(f"Selected Class B: Crocodile")

    # 2. Helper function to collect 20 images from a specific class
    def get_images_from_class(class_name, count=20):
        class_path = os.path.join(dataset_root, class_name, 'images')
        images = []
        tensors = []

        # Collect file names (filtering for .jpg/.JPEG)
        fnames = [f for f in os.listdir(class_path) if f.endswith('.JPEG') or f.endswith('.jpg')]
        fnames = fnames[:count] # Take only the first 20 images

        for fname in fnames:
            path = os.path.join(class_path, fname)
            img = Image.open(path).convert('RGB')

            # Apply transformation (Crucial: use test_transforms to get a clean image)
            if transform:
                img_tensor = transform(img)
            else:
                img_tensor = transforms.ToTensor()(img)

            images.append(img)
            tensors.append(img_tensor)

        return images, torch.stack(tensors)

    # Load the images
    imgs_a, tensors_a = get_images_from_class(Fish)
    imgs_b, tensors_b = get_images_from_class(Crocodile)

    # Combine into one large batch of 40 images
    all_tensors = torch.cat([tensors_a, tensors_b]).to(device)

    # 3. Calculate Embeddings
    with torch.no_grad():
        embeddings = model(all_tensors)
        # Handle cases where model returns a tuple (e.g., output, hidden)
        if isinstance(embeddings, tuple):
            embeddings = embeddings[0]

    # 4. Run t-SNE
    # Perplexity must be low because we only have 40 points in total
    tsne = TSNE(n_components=2, perplexity=10, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings.cpu().numpy())

    # 5. Plotting
    plt.figure(figsize=(10, 8))

    # Plot the first class (first 20 points) in RED
    plt.scatter(embeddings_2d[:20, 0], embeddings_2d[:20, 1], c='red', label=f'Class Fish', s=100, alpha=0.7)

    # Plot the second class (last 20 points) in BLUE
    plt.scatter(embeddings_2d[20:, 0], embeddings_2d[20:, 1], c='blue', label=f'Class Crocodile', s=100, alpha=0.7)

    plt.legend()
    plt.title(f"Separation between Fish and Crocodile")
    plt.grid(True, alpha=0.3)
    plt.show()

# --- Execution ---
# Note: We use TRAIN_PATH because the task requested "from the train set".
# We use test_transforms to ensure images are not randomly cropped or flipped during visualization.
plot_two_classes_separation(model, TRAIN_PATH, test_transforms, device)

Yes, the model is successfully able to separate the two classes. The t-SNE visualization reveals a clear spatial distinction, with the 'Fish' embeddings clustered primarily on the left and the 'Crocodile' embeddings on the right, separated by a noticeable gap. This distinct clustering indicates that the model has learned robust, discriminative feature representations in its high-dimensional latent space, effectively distinguishing the semantic differences between the two categories.

#### 2. ####

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

def show_nearest_neighbors(model, test_loader, device, num_queries=3, k_neighbors=5):
    """
    Finds and visualizes the nearest neighbors for a few query images from a batch.

    Args:
        model: The trained SimCLR model.
        test_loader: DataLoader for the test set.
        device: 'cuda' or 'cpu'.
        num_queries: Number of random images to select as queries (rows in the plot).
        k_neighbors: Number of closest images to find for each query (columns in the plot).
    """
    model.eval()

    # 1. Get a single batch of data
    try:
        images, images_transform = next(iter(test_loader))
    except StopIteration:
        print("Loader is empty.")
        return

    # Check if the batch size is sufficient
    # We need at least k_neighbors + 1 images (the query itself + k neighbors)
    if len(images) < k_neighbors + 1:
        print(f"Batch size {len(images)} is too small for finding {k_neighbors} neighbors.")
        return

    images_transform = images_transform.to(device)

    # 2. Calculate Embeddings for the entire batch
    with torch.no_grad():
        embeddings = model(images_transform)
        # Handle cases where the model returns a tuple (e.g., (embeddings, logits))
        if isinstance(embeddings, tuple):
            embeddings = embeddings[0]

    # 3. Select 'Query' images
    # We simply pick the first 'num_queries' images from the batch to serve as our anchors
    query_indices = range(num_queries)

    # Create the plot: Rows = Queries, Columns = Query Image + Neighbors
    fig, axes = plt.subplots(num_queries, k_neighbors + 1, figsize=(15, 3 * num_queries))

    for i, query_idx in enumerate(query_indices):
        query_emb = embeddings[query_idx]

        # Calculate distances (MSE) between the query and all other images in the batch
        # Formula: (x - y)^2
        dists = torch.mean((embeddings - query_emb.unsqueeze(0))**2, dim=1)

        # Find indices of the images with the smallest distances (sorting)
        # We select k+1 because the closest match (distance 0) is the image itself
        closest_indices = torch.argsort(dists)[:k_neighbors+1]

        # --- Visualization ---

        # First column: The Query Image (Anchor)
        ax_query = axes[i, 0]
        img_show = images[query_idx].permute(1, 2, 0).numpy()
        img_show = np.clip(img_show, 0, 1) # Ensure pixel values are valid
        ax_query.imshow(img_show)
        ax_query.set_title("Query Image")
        ax_query.axis('off')

        # Remaining columns: The Nearest Neighbors
        # We skip index 0 in closest_indices because it is the query image itself
        for j, neighbor_idx in enumerate(closest_indices[1:]):
            ax_neighbor = axes[i, j + 1]

            neighbor_img = images[neighbor_idx].permute(1, 2, 0).numpy()
            neighbor_img = np.clip(neighbor_img, 0, 1)

            # Get the specific distance value
            dist_val = dists[neighbor_idx].item()

            ax_neighbor.imshow(neighbor_img)
            ax_neighbor.set_title(f"Neighbor {j+1}\nMSE: {dist_val:.4f}")
            ax_neighbor.axis('off')

    plt.tight_layout()
    plt.show()

# Example usage:
# Ensure test_loader has a sufficiently large batch_size (e.g., 64)
show_nearest_neighbors(model, test_loader, device)

- **Row 1 (Query: Keyboard & Mouse):**

Result: The model successfully found 3 other keyboards (Neighbors 1, 2, 4).

The "Misses": Neighbor 3 is a periodic table, and Neighbor 5 is a landscape.

Why it happened: The periodic table shares the grid-like texture of the keyboard keys. The landscape is an outlier.


- **Row 2 (Query: Ornate Church Interior):**

Result: It failed to find similar architectural interiors.

The "Misses": It returned forests (Neighbors 1, 2, 4) and a comic book cover (Neighbor 3).

Why it happened: The query image is visually "busy" with high-frequency details and varied colors. The model matched it with other "busy" images (forest leaves, cluttered comic book) rather than understanding the concept of a "building."



- **Row 3 (Query: Goose/Duck):**

Result: It found one semantically correct match: another duck in water (Neighbor 2).


The "Misses": People on grass, a bison on grass, and a lighthouse.


Why it happened: The model seems to be matching the background (green grass/blue water) rather than the foreground object (the bird).
